# Обучение DL-модели

В качетсве основгого фреймворка используем TenserFlow (Keras), для обучения, потому что:
- в библиотеке при обучении сразу считаются ключевые метрики, что упростит логирование
- проще экспериментировать с настройками моделей

## Устанавливаем и импортируем необходимые библиотеки

In [151]:
# !pip install clearml

In [152]:
# !pip install tensorflow

In [153]:
import numpy as np
import pandas as pd

import logging
from clearml import Task


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import mean_absolute_error

## Включаем логирование

По умолчанию будем использовать уровень INFO

In [154]:
logging.basicConfig(
    level = logging.INFO
)

**Подключим логирование в ClearML**

In [ ]:
task_1 = Task.init(
    project_name = 'HSE-GP5',
    task_name = 'FCN_baseline_BIG_DROPOUT_RELU_v9',
    reuse_last_task_id = False
)

ClearML Task: created new task id=223638dbc97d48509d4b2c5e74c92704


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/e510d55020b94b9d8c50a57f8079f5b1/experiments/223638dbc97d48509d4b2c5e74c92704/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Зададим конфикурацию для экспериментов с FCN-моделями

In [ ]:
config_to_experiments = {
    'activation': 'relu',
    'dropout': 0.5,
    'hidden_layers': [512, 256, 64],
    'learning_rate': 1e-3,
    'epochs': 50,
    'batch_size': 512,
    'embedding_dim': 32
}

Передаем конфигурацию в ClearML:

In [157]:
task_1.connect(config_to_experiments)

{'activation': 'relu',
 'dropout': 0.0,
 'hidden_layers': [512, 256, 64],
 'learning_rate': 0.001,
 'epochs': 50,
 'batch_size': 512,
 'embedding_dim': 32}

### Загрузка датасетов, полученных после EDA и предобработки

In [158]:
logging.info('....начало загрузки датасетов....')

INFO:root:....начало загрузки датасетов....


In [159]:
df_train = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_train.csv')
df_val = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_val.csv')
df_test = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_test.csv')

In [160]:
logging.info('....загрузка датасетов закончена....')

INFO:root:....загрузка датасетов закончена....


**Определимся с назначением каждого из датасетов**

Предсказываем 'popularity': 0-100

In [161]:
all_columns = df_train.columns.to_list()
all_columns.remove('Unnamed: 0')
all_columns

['popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'artists_amount',
 'new_explicit',
 'track_genre_acoustic',
 'track_genre_afrobeat',
 'track_genre_alt-rock',
 'track_genre_alternative',
 'track_genre_ambient',
 'track_genre_anime',
 'track_genre_black-metal',
 'track_genre_bluegrass',
 'track_genre_blues',
 'track_genre_brazil',
 'track_genre_breakbeat',
 'track_genre_british',
 'track_genre_cantopop',
 'track_genre_chicago-house',
 'track_genre_children',
 'track_genre_chill',
 'track_genre_classical',
 'track_genre_club',
 'track_genre_comedy',
 'track_genre_country',
 'track_genre_dance',
 'track_genre_dancehall',
 'track_genre_death-metal',
 'track_genre_deep-house',
 'track_genre_detroit-techno',
 'track_genre_disco',
 'track_genre_disney',
 'track_genre_drum-and-bass',
 'track_genre_dub',
 'track_genre_dubstep',
 'track

In [162]:
df_train.drop(columns = 'Unnamed: 0', inplace = True)
df_val.drop(columns = 'Unnamed: 0', inplace = True)
df_test.drop(columns = 'Unnamed: 0', inplace = True)

In [163]:
PREDICT_VAR = 'popularity'
all_columns.remove('popularity')

ARTIST_NAME  = 'artist_label'
all_columns.remove('artist_label')

FEATURES_COLUMNS = all_columns

Теперь определение датасетов

In [164]:
X_train = df_train[FEATURES_COLUMNS].astype(float).values
X_val = df_val[FEATURES_COLUMNS].astype(float).values
X_test = df_test[FEATURES_COLUMNS].astype(float).values

In [165]:
Y_train = df_train[PREDICT_VAR].values
Y_val = df_val[PREDICT_VAR].values
Y_test = df_test[PREDICT_VAR].values

Отдельно обрабатываем столбец с именем артиста, поскольку ...

In [166]:
ART_train = df_train[ARTIST_NAME].values
ART_val = df_val[ARTIST_NAME].values
ART_test = df_test[ARTIST_NAME].values

## Строим DL-модель

Используем 2 входа Keras, поскольку в датасете есть столбцы закодированы 2 принципиально разными способома:
- OHE
- LabelEncoding

https://www.kaggle.com/code/hireme/two-inputs-neural-network-using-keras

In [167]:
numbers_input = keras.Input(shape = (X_train.shape[1],), name = 'numbers_feautures_input')

artists_input = keras.Input(shape = (1,), name = 'artists_feauture_input')

amount_of_artist_max = max(
    df_train['artist_label'].max(),
    df_val['artist_label'].max(),
    df_test['artist_label'].max()
) + 1

artist_embedding = layers.Embedding(
    input_dim = amount_of_artist_max,
    output_dim = config_to_experiments['embedding_dim'],
    name = 'artist_embedding'
)(artists_input)

artist_embedding = layers.Flatten()(artist_embedding)


x_full = layers.Concatenate()([numbers_input, artist_embedding])

**Создадим полносвязные слои для нейросети**

In [168]:
for layer in config_to_experiments['hidden_layers']:
    x_full = layers.Dense(
        layer,
        activation = config_to_experiments['activation']
    )(x_full)

    x_full = layers.Dropout(
        config_to_experiments['dropout']
    )(x_full)

output_data = layers.Dense(1, name = 'output_data')(x_full)

**Создадим сам объект модели**

In [169]:
model_task_1 = keras.Model(
    inputs = [numbers_input, artists_input],
    outputs = output_data
)

In [170]:
model_task_1.compile(
    optimizer = keras.optimizers.Adam(learning_rate = config_to_experiments['learning_rate']),
    loss = 'mse',
    metrics = ['mae']
)

Проверим, что все параметры корректны:

In [171]:
model_task_1.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ artists_feauture_i… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ artist_embedding    │ (None, 1, 32)     │    486,912 │ artists_feauture… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numbers_feautures_… │ (None, 130)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 32)        │          0 │ artist_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 162)       │          0 │ numbers_feauture… │
│ (Concatenate)       │                   │            │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 512)       │     83,456 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 512)       │          0 │ dense_16[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 256)       │    131,328 │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 256)       │          0 │ dense_17[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 64)        │     16,448 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 64)        │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_data (Dense) │ (None, 1)         │         65 │ dropout_18[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 718,209 (2.74 MB)

 Trainable params: 718,209 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

## Обучение модели

In [172]:
logging.info('Начало обучения модели')
logger = task_1.get_logger()

INFO:root:Начало обучения модели


In [ ]:
learning_history = model_task_1.fit(
    [X_train, ART_train], y = Y_train,
    validation_data = ([X_val, ART_val], Y_val),
    epochs = config_to_experiments['epochs'],
    batch_size = config_to_experiments['batch_size'],
    verbose = 1
)

Epoch 1/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 441.1624 - mae: 15.5181 - val_loss: 223.2800 - val_mae: 10.2102
Epoch 2/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 176.0847 - mae: 8.7041 - val_loss: 208.7980 - val_mae: 9.4097
Epoch 3/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 151.0394 - mae: 7.7380 - val_loss: 206.7978 - val_mae: 9.1904
Epoch 4/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 142.3244 - mae: 7.3448 - val_loss: 205.3777 - val_mae: 9.0853
Epoch 5/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 137.6125 - mae: 7.1304 - val_loss: 205.3415 - val_mae: 9.0551
Epoch 6/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 134.4629 - mae: 6.9778 - val_loss: 205.2422 - val_mae: 9.0513
Epoch 7/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 132.0317 - mae: 6.8675 - val_loss: 204.1099 - val_mae: 8.9981
Epoch 8/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 129.6925 - mae: 6.7637 - val_loss: 203.7645 - val_mae: 9.0000
Epoch 9/50
123/123 ━

Сделаем логирование по эпохам обучения

In [ ]:
for epoch, (train_mae_metric, val_mae_metric) in enumerate(zip(
    learning_history.history['mae'],
    learning_history.history['val_mae']
    )):

    logger.report_scalar('MAE', 'train', value = train_mae_metric, iteration = epoch)
    logger.report_scalar('MAE', 'val', value = val_mae_metric, iteration = epoch)

In [ ]:
for epoch, (train_loss, val_loss) in enumerate(zip(
    learning_history.history['loss'],
    learning_history.history['val_loss']
    )):

    logger.report_scalar('MSE loss', 'train', value = train_loss, iteration = epoch)
    logger.report_scalar('MSE loss', 'val', value = val_loss, iteration = epoch)

In [ ]:
logging.info('Проверка точности модели на test')

INFO:root:Проверка точности модели на test


In [ ]:
y_pred = model_task_1.predict(
    [X_test, ART_test]
).flatten()

mae_mark_test = mean_absolute_error(Y_test, y_pred)
rmse_mark_test = np.sqrt(np.mean((Y_test - y_pred)**2))

421/421 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
logging.info(f'\nTEST_MAE = {mae_mark_test}\nTEST_RMSE = {rmse_mark_test}')

logger.report_single_value(name = 'TEST_MAE', value = mae_mark_test)
logger.report_single_value(name = 'TEST_RMSE', value = rmse_mark_test)

INFO:root:
TEST_MAE = 8.688563346862793
TEST_RMSE = 14.073923941844207


## Сохранение моделей

In [ ]:
NAME = "FCN_baseline_ACTIVATION" + str(8)
PATH = f'/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/DL/FCN_models_task_1/{NAME}.keras'

model_task_1.save(PATH)

In [ ]:
task_1.update_output_model(
    model_path = PATH,
    model_name = NAME
)

logging.info(f'Обучение модели {NAME} завершено успешно')
task_1.close()

INFO:root:Обучение модели FCN_baseline_ACTIVATION7 завершено успешно
█████████████████████████████████ 100% | 8.27/8.27 MB [00:01<00:00,  4.33MB/s]: 
